# 03 — Job Satisfaction Analysis

Compare job satisfaction across roles, countries, work arrangements, pay levels, and AI tool usage using SO 2025 (JobSat) and SO 2023 (AIBen).

In [1]:
import os, warnings
warnings.filterwarnings("ignore")
os.chdir("/app")
import pathlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA_PROC = pathlib.Path("data/processed")


In [2]:
so25 = pd.read_parquet(DATA_PROC / "so25_processed.parquet")
so23 = pd.read_parquet(DATA_PROC / "so23_processed.parquet")

# Derive primary role
so25["DevType_primary"] = so25["DevType"].str.split(";").str[0].str.strip()
so23["DevType_primary"] = so23["DevType"].str.split(";").str[0].str.strip()

print(f"SO 2025: {so25.shape} — JobSat non-null: {so25['JobSat'].notna().sum():,}")
print(f"SO 2023: {so23.shape} — AIBen non-null: {so23['AIBen'].notna().sum():,}")
print("\nJobSat scale (SO 2025):", sorted(so25["JobSat"].dropna().unique()))


SO 2025: (13739, 175) — JobSat non-null: 11,522
SO 2023: (28648, 85) — AIBen non-null: 19,591

JobSat scale (SO 2025): [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]


## Q1: Which roles have highest job satisfaction?

In [3]:
# SO 2025 JobSat by role (top roles with ≥30 respondents)
role_sat = (
    so25[so25["JobSat"].notna()]
    .groupby("DevType_primary")["JobSat"]
    .agg(mean="mean", median="median", count="count")
    .query("count >= 30")
    .sort_values("median", ascending=True)
    .round(2)
)
print("Job satisfaction by role (SO 2025, JobSat 1-10):")
print(role_sat.to_string())

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(role_sat.index, role_sat["median"], color="#4C72B0")
ax.axvline(so25["JobSat"].median(), color="#DD4444", linestyle="--",
           label=f"Overall median: {so25['JobSat'].median():.1f}")
ax.set_xlabel("Median Job Satisfaction (1-10)")
ax.set_title("Median Job Satisfaction by Role — SO 2025")
ax.legend()
plt.tight_layout()
plt.savefig("figures/fig_jobsat_by_role.png", dpi=120, bbox_inches="tight")
plt.show()


Job satisfaction by role (SO 2025, JobSat 1-10):
                                               mean  median  count
DevType_primary                                                   
Academic researcher                            7.35     7.0     86
Cloud infrastructure engineer                  7.21     7.0    118
Cybersecurity or InfoSec professional          7.51     7.0     47
Developer, mobile                              7.13     7.0    371
Data or business analyst                       7.06     7.0     31
Database administrator or engineer             6.95     7.0     40
Developer, front-end                           7.13     7.0    579
Developer, QA or test                          6.52     7.0     81
Developer, back-end                            7.04     7.0   2202
Developer, desktop or enterprise applications  7.12     7.0    674
Developer, embedded applications or devices    7.01     7.0    463
AI/ML engineer                                 7.22     8.0    183
Founder, tech

## Q2: Do higher paid developers report more satisfaction?

In [4]:
so25_sat = so25[so25["JobSat"].notna() & so25["CompUSD"].notna()].copy()

# Salary bands
so25_sat["comp_band"] = pd.cut(
    so25_sat["CompUSD"],
    bins=[0, 50_000, 100_000, 150_000, 200_000, 400_001],
    labels=["<$50K", "$50-100K", "$100-150K", "$150-200K", ">$200K"],
)
band_sat = (
    so25_sat.groupby("comp_band", observed=True)["JobSat"]
    .agg(mean="mean", median="median", count="count")
    .round(2)
)
print("Job satisfaction by salary band:")
print(band_sat.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(band_sat.index, band_sat["median"], color=["#6baed6","#3182bd","#2171b5","#08519c","#08306b"])
ax.set_xlabel("Salary Band")
ax.set_ylabel("Median Job Satisfaction (1-10)")
ax.set_title("Job Satisfaction vs Salary Band — SO 2025")
plt.tight_layout()
plt.savefig("figures/fig_jobsat_by_salary.png", dpi=120, bbox_inches="tight")
plt.show()

corr = so25_sat[["CompUSD","JobSat"]].corr().iloc[0,1]
print(f"\nPearson correlation CompUSD ~ JobSat: {corr:.3f}")


Job satisfaction by salary band:
           mean  median  count
comp_band                     
<$50K      6.96     7.0   2536
$50-100K   7.17     7.0   4487
$100-150K  7.31     8.0   2313
$150-200K  7.34     8.0   1254
>$200K     7.50     8.0    932



Pearson correlation CompUSD ~ JobSat: 0.077


## Q3: Do AI tool users report higher satisfaction?

In [5]:
# SO 2025 AISelect categories
print("AISelect values (SO 2025):")
print(so25["AISelect"].value_counts().to_string())

ai_sat = (
    so25[so25["AISelect"].notna() & so25["JobSat"].notna()]
    .groupby("AISelect")["JobSat"]
    .agg(mean="mean", median="median", count="count")
    .round(2)
)
print("\nJob satisfaction by AI tool usage:")
print(ai_sat.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
labels = [l[:40] for l in ai_sat.index]
ax.bar(range(len(labels)), ai_sat["median"], color="#55A868")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Median Job Satisfaction (1-10)")
ax.set_title("Job Satisfaction by AI Tool Usage — SO 2025")
plt.tight_layout()
plt.savefig("figures/fig_jobsat_ai_usage.png", dpi=120, bbox_inches="tight")
plt.show()


AISelect values (SO 2025):
AISelect
Yes, I use AI tools daily                      6155
Yes, I use AI tools weekly                     2558
No, and I don't plan to                        2151
Yes, I use AI tools monthly or infrequently    1932
No, but I plan to soon                          624

Job satisfaction by AI tool usage:
                                             mean  median  count
AISelect                                                        
No, and I don't plan to                      7.05     7.0   1743
No, but I plan to soon                       7.09     8.0    487
Yes, I use AI tools daily                    7.31     8.0   5382
Yes, I use AI tools monthly or infrequently  7.09     7.0   1575
Yes, I use AI tools weekly                   7.14     7.0   2095


In [6]:
# SO 2023: AIBen (trust in AI)
if "AIBen" in so23.columns:
    print("AIBen values (SO 2023):")
    print(so23["AIBen"].value_counts().to_string())
else:
    print("AIBen not in SO 2023 processed data")


AIBen values (SO 2023):
AIBen
Somewhat trust                7004
Neither trust nor distrust    6068
Somewhat distrust             4922
Highly distrust               1235
Highly trust                   362


## Satisfaction by Country and Remote Work

In [7]:
# By country (top 10 by response count)
top_countries_sat = so25["Country"].value_counts().head(10).index.tolist()
country_sat = (
    so25[so25["JobSat"].notna() & so25["Country"].isin(top_countries_sat)]
    .groupby("Country")["JobSat"]
    .agg(mean="mean", median="median", count="count")
    .sort_values("median", ascending=False)
    .round(2)
)
print("Satisfaction by country (top 10):")
print(country_sat.to_string())


Satisfaction by country (top 10):
                                                      mean  median  count
Country                                                                  
Brazil                                                7.43     8.0    368
Canada                                                7.31     8.0    629
Netherlands                                           7.30     8.0    408
Spain                                                 7.19     8.0    374
United States of America                              7.32     8.0   3567
France                                                7.14     7.0    669
Germany                                               6.91     7.0   1448
India                                                 7.10     7.0    540
Italy                                                 6.90     7.0    376
United Kingdom of Great Britain and Northern Ireland  7.00     7.0   1059


In [8]:
# By RemoteWork
remote_sat = (
    so25[so25["JobSat"].notna() & so25["RemoteWork"].notna()]
    .groupby("RemoteWork")["JobSat"]
    .agg(mean="mean", median="median", count="count")
    .sort_values("median", ascending=False)
    .round(2)
)
print("Satisfaction by work arrangement:")
print(remote_sat.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
remote_sat_plot = remote_sat.sort_values("median")
ax.barh(range(len(remote_sat_plot)), remote_sat_plot["median"], color="#4C72B0")
ax.set_yticks(range(len(remote_sat_plot)))
ax.set_yticklabels([l[:45] for l in remote_sat_plot.index], fontsize=8)
ax.set_xlabel("Median Job Satisfaction (1-10)")
ax.set_title("Satisfaction by Work Arrangement — SO 2025")
plt.tight_layout()
plt.savefig("figures/fig_jobsat_remote.png", dpi=120, bbox_inches="tight")
plt.show()


Satisfaction by work arrangement:
                                                                              mean  median  count
RemoteWork                                                                                       
Remote                                                                        7.29     8.0   4356
Your choice (very flexible, you can come in when you want or just as needed)  7.32     8.0   1507
Hybrid (some in-person, leans heavy to flexibility)                           7.17     7.0   2179
Hybrid (some remote, leans heavy to in-person)                                7.05     7.0   2205
In-person                                                                     7.01     7.0   1222


## Save Summary

In [9]:
# Combine into one summary table (role-level stats)
summary_role = (
    so25[so25["JobSat"].notna()]
    .groupby("DevType_primary")
    .agg(
        mean_jobsat=("JobSat", "mean"),
        median_jobsat=("JobSat", "median"),
        count_jobsat=("JobSat", "count"),
        median_salary=("CompUSD", lambda x: x.median() if x.notna().any() else np.nan),
    )
    .reset_index()
    .rename(columns={"DevType_primary": "role"})
    .round(2)
)

# AI usage summary
ai_usage_summary = (
    so25[so25["AISelect"].notna() & so25["JobSat"].notna()]
    .groupby("AISelect")
    .agg(
        mean_jobsat=("JobSat", "mean"),
        median_jobsat=("JobSat", "median"),
        count=("JobSat", "count"),
    )
    .reset_index()
    .rename(columns={"AISelect": "ai_usage"})
    .round(2)
)

# Remote work summary
remote_summary = (
    so25[so25["JobSat"].notna() & so25["RemoteWork"].notna()]
    .groupby("RemoteWork")
    .agg(
        mean_jobsat=("JobSat", "mean"),
        median_jobsat=("JobSat", "median"),
        count=("JobSat", "count"),
    )
    .reset_index()
    .rename(columns={"RemoteWork": "remote_work"})
    .round(2)
)

# SO 2023 AIBen
if "AIBen" in so23.columns:
    aiben_summary = (
        so23[so23["AIBen"].notna()]
        .groupby("AIBen")
        .agg(
            count=("AIBen", "count"),
            median_salary=("CompUSD", "median"),
        )
        .reset_index()
        .rename(columns={"AIBen": "ai_ben"})
    )
else:
    aiben_summary = pd.DataFrame(columns=["ai_ben","count","median_salary"])

# Save
summary_role.to_parquet(DATA_PROC / "job_satisfaction_summary.parquet", index=False)
ai_usage_summary.to_parquet(DATA_PROC / "ai_usage_satisfaction.parquet", index=False)
remote_summary.to_parquet(DATA_PROC / "remote_satisfaction.parquet", index=False)
aiben_summary.to_parquet(DATA_PROC / "aiben_summary.parquet", index=False)

print(f"Saved job_satisfaction_summary.parquet — {len(summary_role)} roles")
print(f"Saved ai_usage_satisfaction.parquet — {len(ai_usage_summary)} AI categories")
print(f"Saved remote_satisfaction.parquet — {len(remote_summary)} work types")
print(f"Saved aiben_summary.parquet — {len(aiben_summary)} AIBen categories")


Saved job_satisfaction_summary.parquet — 31 roles
Saved ai_usage_satisfaction.parquet — 5 AI categories
Saved remote_satisfaction.parquet — 5 work types
Saved aiben_summary.parquet — 5 AIBen categories
